# Natural Gas Price Prediction with Qwen3 LLM

This notebook uses Qwen3 LLM to predict next-day Henry Hub Natural Gas Spot Prices based on news headlines and current prices.

## Imports

In [ ]:
import requests
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Dict, Any, List, Tuple, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from datetime import datetime, timedelta
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# API Configuration
API_ENDPOINT = "https://khh6fxbehjar.share.zrok.io/v1/chat/completions"
MODEL_NAME = "hoangquan456/qwen3-nothink:1.7b"

# Data files
HEADLINES_FILE = "gs://codeml/news_filtered_combined.json"
PRICES_FILE = "gs://codeml/DHHNGSP.csv"

# Processing parameters
MAX_WORKERS = 6
REQUEST_TIMEOUT = 10
MAX_RETRIES = 3
INITIAL_RETRY_DELAY = 1

# Date range for testing (matching FinBERT: 2020-01-01 onwards)
TEST_START_DATE = '2020-01-01'

print("Configuration:")
print(f"  API: {API_ENDPOINT}")
print(f"  Model: {MODEL_NAME}")
print(f"  Test start date: {TEST_START_DATE}")
print(f"  Max workers: {MAX_WORKERS}")

## Load Data

In [ ]:
# Load price data
prices = pd.read_csv(PRICES_FILE)
prices['observation_date'] = pd.to_datetime(prices['observation_date'])
prices = prices.dropna()

# Calculate next day return
prices['next_day_price'] = prices['DHHNGSP'].shift(-1)
prices['next_day_return'] = prices['DHHNGSP'].pct_change().shift(-1)
prices = prices.dropna()

print(f"Price data: {len(prices)} days")
print(f"Date range: {prices['observation_date'].min()} to {prices['observation_date'].max()}")
print(f"\nPrice statistics:")
print(prices['DHHNGSP'].describe())

In [ ]:
# Load headlines
headlines = pd.read_json(HEADLINES_FILE)
headlines['date'] = pd.to_datetime(headlines['date'])

print(f"Headlines: {len(headlines)} articles")
print(f"Date range: {headlines['date'].min()} to {headlines['date'].max()}")
print(f"Unique days: {headlines['date'].nunique()}")

## Merge Headlines with Prices

In [ ]:
# Merge headlines with price data
merged = pd.merge(
    prices,
    headlines,
    how='left',
    left_on='observation_date',
    right_on='date'
).drop(columns=['date'])

merged = merged.dropna()

# Filter for test period
test_data = merged[merged['observation_date'] >= TEST_START_DATE].copy()

print(f"\nTest data: {len(test_data)} headline-day pairs")
print(f"Unique days: {test_data['observation_date'].nunique()}")
print(f"Date range: {test_data['observation_date'].min()} to {test_data['observation_date'].max()}")

## Aggregate Headlines by Day

In [ ]:
# Aggregate all headlines per day
test_daily = test_data.groupby('observation_date').agg({
    'headline': lambda x: '\n'.join([f"- {h}" for h in x]),
    'summary': lambda x: '\n'.join([f"{s}" for s in x if pd.notna(s)]),
    'DHHNGSP': 'first',
    'next_day_price': 'first',
    'next_day_return': 'first'
}).reset_index()

print(f"Aggregated to {len(test_daily)} days for testing")
print(f"\nSample aggregated data:")
print(test_daily[['observation_date', 'DHHNGSP', 'next_day_price']].head())

## Qwen3 Price Prediction Function

In [ ]:
def predict_price_with_qwen(current_price: float, headlines: str, summaries: str, date: str) -> Optional[Dict[str, Any]]:
    """
    Predict next-day natural gas price using Qwen3 LLM.
    
    Args:
        current_price: Current Henry Hub Natural Gas Spot Price
        headlines: Concatenated headlines for the day
        summaries: Concatenated summaries for the day
        date: Current date string
        
    Returns:
        Dict with 'direction' (up/down/neutral) and 'difference' (dollar amount change)
        Returns None if prediction fails
    """
    
    system_prompt = """You are a natural gas market analyst specializing in Henry Hub Natural Gas Spot Price predictions.
Your task is to predict the next day's price movement based on current news and today's price.

You must respond with ONLY a JSON object in this exact format:
{"direction": "up" or "down" or "neutral", "difference": <dollar_change>}

Where:
- direction: "up" if price will increase, "down" if decrease, "neutral" if no significant change
- difference: predicted dollar amount change from today's price
  * Positive value for price increases (e.g., 0.15 means +$0.15)
  * Negative value for price decreases (e.g., -0.20 means -$0.20)
  * MUST be 0 if direction is "neutral"

Example outputs:
{"direction": "up", "difference": 0.15}
{"direction": "down", "difference": -0.20}
{"direction": "neutral", "difference": 0}

Return ONLY the JSON object with no additional text or markdown formatting."""
    
    user_prompt = f"""Analyze today's natural gas market news and predict tomorrow's Henry Hub Natural Gas Spot Price.

DATE: {date}
CURRENT PRICE: ${current_price:.2f}

TODAY'S HEADLINES:
{headlines}

ARTICLE SUMMARIES:
{summaries[:1000] if summaries else 'No summaries available'}

Based on this information, predict tomorrow's price change in DOLLAR AMOUNT.
Consider:
- Supply/demand dynamics
- Weather impacts
- Storage levels
- Market sentiment
- Geopolitical factors

IMPORTANT: 
- If direction is "neutral", difference MUST be 0
- Return the dollar amount difference, not percentage
- Example: if current price is $2.50 and you predict $2.65, return {{"direction": "up", "difference": 0.15}}

Respond with ONLY the JSON object:"""
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.3  # Lower temperature for more consistent predictions
    }
    
    # Retry loop
    for attempt in range(MAX_RETRIES):
        try:
            response = requests.post(
                API_ENDPOINT,
                headers={"Content-Type": "application/json"},
                json=payload,
                timeout=REQUEST_TIMEOUT
            )
            response.raise_for_status()
            
            # Parse response
            assistant_message = response.json()['choices'][0]['message']['content'].strip()
            
            # Remove markdown formatting if present
            if assistant_message.startswith('```json'):
                assistant_message = assistant_message[7:]
            if assistant_message.startswith('```'):
                assistant_message = assistant_message[3:]
            if assistant_message.endswith('```'):
                assistant_message = assistant_message[:-3]
            assistant_message = assistant_message.strip()
            
            # Parse JSON
            result = json.loads(assistant_message)
            
            # Validate required fields
            if 'direction' in result and 'difference' in result:
                # Enforce neutral direction has difference = 0
                if result['direction'] == 'neutral':
                    result['difference'] = 0
                return result
            else:
                if attempt < MAX_RETRIES - 1:
                    continue
                return None
                
        except requests.exceptions.Timeout:
            if attempt < MAX_RETRIES - 1:
                time.sleep(INITIAL_RETRY_DELAY * (2 ** attempt))
                continue
            return None
            
        except requests.exceptions.HTTPError as e:
            if attempt < MAX_RETRIES - 1 and e.response.status_code in [504, 502, 503]:
                time.sleep(INITIAL_RETRY_DELAY * (2 ** attempt))
                continue
            return None
            
        except (json.JSONDecodeError, KeyError, ValueError):
            if attempt < MAX_RETRIES - 1:
                time.sleep(INITIAL_RETRY_DELAY * (2 ** attempt))
                continue
            return None
            
        except Exception as e:
            return None
    
    return None

print("✓ Qwen3 prediction function defined")

## Run Predictions with Concurrent Processing

In [ ]:
def process_single_prediction(row: pd.Series) -> Dict[str, Any]:
    """Process a single day's prediction."""
    prediction = predict_price_with_qwen(
        current_price=row['DHHNGSP'],
        headlines=row['headline'],
        summaries=row['summary'],
        date=row['observation_date'].strftime('%Y-%m-%d')
    )
    
    return {
        'date': row['observation_date'],
        'current_price': row['DHHNGSP'],
        'actual_next_price': row['next_day_price'],
        'actual_return': row['next_day_return'],
        'prediction': prediction
    }

print("="*80)
print("Running Qwen3 Predictions")
print("="*80)

results = []
failed_count = 0

# Use ThreadPoolExecutor for concurrent predictions
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_single_prediction, row): idx 
               for idx, row in test_daily.iterrows()}
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Predicting"):
        try:
            result = future.result()
            if result['prediction'] is not None:
                results.append(result)
            else:
                failed_count += 1
        except Exception as e:
            failed_count += 1

print(f"\nPrediction Results:")
print(f"  Successful: {len(results)}")
print(f"  Failed: {failed_count}")
print(f"  Success rate: {len(results)/(len(results)+failed_count)*100:.1f}%")

## Process Predictions and Calculate Returns

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Extract prediction values
results_df['predicted_difference'] = results_df['prediction'].apply(lambda x: x['difference'])
results_df['predicted_direction'] = results_df['prediction'].apply(lambda x: x['direction'])

# Calculate predicted next price using difference
# Formula: predicted_next_price = current_price + difference
results_df['predicted_next_price'] = results_df['current_price'] + results_df['predicted_difference']

# Calculate predicted return for comparison
results_df['predicted_return'] = results_df['predicted_difference'] / results_df['current_price']

# Calculate errors
results_df['return_error'] = results_df['predicted_return'] - results_df['actual_return']
results_df['price_error'] = results_df['predicted_next_price'] - results_df['actual_next_price']
results_df['price_error_pct'] = (results_df['price_error'] / results_df['actual_next_price']) * 100

print(f"\nProcessed {len(results_df)} predictions")
print(f"\nSample predictions:")
print(results_df[['date', 'current_price', 'predicted_difference', 'predicted_next_price', 'actual_next_price', 'predicted_direction']].head(10))

## Evaluation Metrics

In [ ]:
print("="*80)
print("QWEN3 MODEL EVALUATION")
print("="*80)

# Return-based metrics
predicted_returns = results_df['predicted_return'].values
actual_returns = results_df['actual_return'].values

mse = ((predicted_returns - actual_returns) ** 2).mean()
rmse = mse ** 0.5
mae = abs(predicted_returns - actual_returns).mean()

# Baseline comparison
naive_pred = np.zeros_like(actual_returns)
naive_rmse = ((naive_pred - actual_returns) ** 2).mean() ** 0.5

# Directional accuracy
predicted_direction = predicted_returns > 0
actual_direction = actual_returns > 0
directional_accuracy = (predicted_direction == actual_direction).mean() * 100

# Information Coefficient
ic, ic_pvalue = spearmanr(predicted_returns, actual_returns)

print(f"\nReturn Prediction Performance:")
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  Normalized Score (1/(1+RMSE)): {(1/(1+rmse)):.4f}")

print(f"\nBaseline Comparison:")
print(f"  Naive (predict 0) RMSE: {naive_rmse:.4f}")
print(f"  Model Improvement: {((naive_rmse - rmse) / naive_rmse * 100):.1f}%")

print(f"\nDirectional & Correlation Metrics:")
print(f"  Directional Accuracy: {directional_accuracy:.2f}%")
print(f"  Information Coefficient (Spearman): {ic:.4f} (p={ic_pvalue:.4f})")

# Price-based metrics
predicted_prices = results_df['predicted_next_price'].values
actual_prices = results_df['actual_next_price'].values

price_mse = ((predicted_prices - actual_prices) ** 2).mean()
price_rmse = price_mse ** 0.5
price_mae = abs(predicted_prices - actual_prices).mean()
price_mape = (abs((actual_prices - predicted_prices) / actual_prices) * 100).mean()

print(f"\nPrice Prediction Performance:")
print(f"  Price RMSE: ${price_rmse:.4f}")
print(f"  Price MAE:  ${price_mae:.4f}")
print(f"  Price MAPE: {price_mape:.2f}%")
print(f"  Price-based Score (1/(1+RMSE)): {(1/(1+price_rmse)):.4f}")

# Volatility regime analysis
results_df['abs_return'] = abs(results_df['actual_return'])
high_vol_threshold = results_df['abs_return'].quantile(0.75)
high_vol_mask = results_df['abs_return'] >= high_vol_threshold
low_vol_mask = ~high_vol_mask

print(f"\nPerformance by Volatility Regime:")
print(f"  High Volatility Days (top 25%):")
high_vol_rmse = ((results_df.loc[high_vol_mask, 'predicted_return'] - results_df.loc[high_vol_mask, 'actual_return'])**2).mean()**0.5
high_vol_dir_acc = ((results_df.loc[high_vol_mask, 'predicted_return'] > 0) == (results_df.loc[high_vol_mask, 'actual_return'] > 0)).mean()*100
print(f"    RMSE: {high_vol_rmse:.4f}")
print(f"    Directional Accuracy: {high_vol_dir_acc:.2f}%")

print(f"  Low Volatility Days (bottom 75%):")
low_vol_rmse = ((results_df.loc[low_vol_mask, 'predicted_return'] - results_df.loc[low_vol_mask, 'actual_return'])**2).mean()**0.5
low_vol_dir_acc = ((results_df.loc[low_vol_mask, 'predicted_return'] > 0) == (results_df.loc[low_vol_mask, 'actual_return'] > 0)).mean()*100
print(f"    RMSE: {low_vol_rmse:.4f}")
print(f"    Directional Accuracy: {low_vol_dir_acc:.2f}%")

## Direction Distribution Analysis

In [ ]:
print("\n" + "="*80)
print("DIRECTION PREDICTION ANALYSIS")
print("="*80)

direction_counts = results_df['predicted_direction'].value_counts()
print(f"\nPredicted Direction Distribution:")
for direction, count in direction_counts.items():
    print(f"  {direction}: {count} ({count/len(results_df)*100:.1f}%)")

# Actual direction distribution
actual_up = (results_df['actual_return'] > 0).sum()
actual_down = (results_df['actual_return'] < 0).sum()
actual_neutral = (results_df['actual_return'] == 0).sum()

print(f"\nActual Direction Distribution:")
print(f"  up: {actual_up} ({actual_up/len(results_df)*100:.1f}%)")
print(f"  down: {actual_down} ({actual_down/len(results_df)*100:.1f}%)")
print(f"  neutral: {actual_neutral} ({actual_neutral/len(results_df)*100:.1f}%)")

## Final Score

In [ ]:
final_score = 1 / (1 + price_rmse)

print("\n" + "="*80)
print(f"FINAL SCORE: {final_score:.4f}")
print("="*80)

print(f"\nScore Breakdown:")
print(f"  Formula: 1 / (1 + Price_RMSE)")
print(f"  Price RMSE: ${price_rmse:.4f}")
print(f"  Score: {final_score:.4f}")

## Save Results

In [ ]:
# Save predictions to CSV
output_df = results_df[[
    'date', 'current_price', 'predicted_next_price', 'actual_next_price',
    'predicted_return', 'actual_return', 'predicted_direction',
    'price_error', 'price_error_pct'
]].copy()

output_file = 'qwen3_predictions.csv'
output_df.to_csv(output_file, index=False)
print(f"\n✓ Predictions saved to: {output_file}")
print(f"✓ Total predictions: {len(output_df)}")